# Class Exercise: Comparing Recommender Models

<small>OPAN 6604. Dataset: MovieLens (`MovieLense-Movies.csv`, `MovieLense-Ratings.csv`) - 610 users, ~9,700 movies, ~100K ratings.</small>

<small>**Task:** The data, train/test split, and the `precision_recall_at_k` helper are set up for you (same as the demo). Run a small **model bake-off**: assemble several candidate models, evaluate them on the held-out test set, and pick which to deploy.

- **A)** Complete the `models` dict - add three candidate CF models to the provided **Baseline**: **UBCF · cosine · k=10**, **UBCF · pearson · k=50**, and **IBCF · cosine · k=50**.
- **B)** Evaluate all four on the test set (RMSE, Precision@10, Recall@10) in one comparison table.
- **C)** Decide which model to deploy for a top-10 list - and explain why the lowest-RMSE model isn't necessarily your pick.</small>

### Step 0: Setup

In [ ]:
import pandas as pd
import numpy as np
from collections import defaultdict
from surprise import KNNBasic, BaselineOnly, Dataset, Reader, accuracy
from surprise.model_selection import train_test_split

# Fix the seed so the split and models are reproducible across runs.
RANDOM_STATE = 6604

: 

### Load data

In [ ]:
# Load the catalog and the ratings; build a movieId -> title lookup for display.
movies = pd.read_csv("MovieLense-Movies.csv")
ratings = pd.read_csv("MovieLense-Ratings.csv")
title_of = dict(zip(movies["movieId"], movies["title"]))

print(f"Movies: {movies.shape}  |  Ratings: {ratings.shape}")

### Build the Surprise dataset & train/test split

<small>Wrap the long-format ratings into a `surprise` dataset (MovieLens uses a 0.5–5.0 scale), then hold out 10% so predictions can be scored against ratings the model never saw.</small>

In [ ]:
reader = Reader(rating_scale=(0.5, 5.0))
data = Dataset.load_from_df(ratings[["userId", "movieId", "rating"]], reader)

trainset, testset = train_test_split(data, test_size=0.1, random_state=RANDOM_STATE)
print(f"Train: {trainset.n_ratings} ratings  |  Test: {len(testset)} ratings")

### Ranking metric helper (provided)

<small>`precision_recall_at_k` is given to you — same as the demo. A movie is "relevant" if its true rating ≥ 4.0; precision is the share of the **top-N** items that were relevant, recall the share of the user's relevant items captured in the top-N. Note `top_n` (the list length) is distinct from the model's `k` (the neighborhood size).</small>

In [ ]:
def precision_recall_at_k(predictions, top_n=10, threshold=4.0):
    user_data = defaultdict(list)
    for uid, _, true_r, est, _ in predictions:
        user_data[uid].append((est, true_r))
    precisions, recalls = [], []
    for items in user_data.values():
        items.sort(key=lambda x: x[0], reverse=True)
        n_relevant = sum(1 for _, t in items if t >= threshold)
        n_hits = sum(1 for _, t in items[:top_n] if t >= threshold)
        precisions.append(n_hits / top_n)
        if n_relevant > 0:
            recalls.append(n_hits / n_relevant)
    return np.mean(precisions), np.mean(recalls)

### A) Define the candidate models

<small>Models are defined directly in a `models` dict, then evaluated together below. The **Baseline** (per-user/item means) is provided as the bar to clear. Add three CF candidates with `KNNBasic`:

1. **UBCF · cosine · k=10** - user-based, cosine similarity
2. **UBCF · pearson · k=50** - user-based, Pearson, 50 neighbors
3. **IBCF · cosine · k=50** - item-based, cosine, 50 neighbors

Hint: `sim_options` takes `"name"` (`"pearson"` / `"cosine"`) and `"user_based"` (`True` / `False`); `k` is a separate `KNNBasic` argument (the neighborhood size).</small>

In [ ]:
models = {
    "Baseline": BaselineOnly(verbose=False),
    "UBCF · cosine · k=10": KNNBasic(
        k=10,
        sim_options={"name": "cosine", "user_based": True},
        verbose=False,
    ),
    "UBCF · pearson · k=50": KNNBasic(
        k=50,
        sim_options={"name": "pearson", "user_based": True},
        verbose=False,
    ),
    "IBCF · cosine · k=50": KNNBasic(
        k=50,
        sim_options={"name": "cosine", "user_based": False},
        verbose=False,
    ),
}

: 

### B) Evaluate every model and compare

<small>Loop over `models`: fit each on `trainset`, predict `testset`, then compute RMSE (`accuracy.rmse`) and Precision@10 & Recall@10 (`precision_recall_at_k`, with `top_n=10`). Collect one row per model and display the comparison with `pd.DataFrame(results).round(4)`.</small>

In [ ]:
# ...

### C) Which model would you deploy?

<small>In a few sentences: which model has the lowest RMSE? Which is best on Precision@10 / Recall@10 — is it the same model? For a top-10 recommendation list, which would you ship, and why?</small>

_Your answer here._